# Notebook 0 — Exploratory Data Analysis (EDA) & Data Understanding

## Project
Targeted Social Prescription: Matching & Outcome Evaluation  
Dataset: RAND HRS Longitudinal File (1992–2022)

---

## Purpose of This Notebook

This notebook performs:

1. Initial data inspection
2. Variable structure examination
3. Missingness assessment
4. Basic distribution checks for key variables

This notebook does NOT:

- Construct the final analytic cohort
- Engineer risk scores
- Generate recommendations
- Perform evaluation

All analytic dataset construction will be conducted in Notebook 1.

---

This EDA notebook is used for:

- Understanding data structure
- Supporting the Data section in the capstone report
- Identifying potential data quality issues

In [19]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

# 定义路径
DATA_RAW_DIR = "../data_raw"
DATA_PROCESSED_DIR = "../data_processed"
FIG_DIR = "../figures"

# 创建目录（如果不存在）
os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Data raw dir: {DATA_RAW_DIR}")
print(f"Data processed dir: {DATA_PROCESSED_DIR}")
print(f"Figures dir: {FIG_DIR}")

Data raw dir: ../data_raw
Data processed dir: ../data_processed
Figures dir: ../figures


In [23]:
# ----------------------------
# Load a small sample safely (do NOT load full .dta)
# ----------------------------
INFILE = os.path.join(DATA_RAW_DIR, "randhrs1992_2022v1.dta")

print(f"Reading data from: {INFILE}")
print(f"File exists: {os.path.exists(INFILE)}")
if not os.path.exists(INFILE):
    raise FileNotFoundError("randhrs1992_2022v1.dta not found in ../data_raw. Please check your path.")

# Read a tiny chunk to get columns + quick preview
it = pd.read_stata(INFILE, iterator=True)
df_sample = it.read(5000)   # small sample for EDA
all_cols = df_sample.columns.tolist()

# close iterator (compat)
if hasattr(it, "close"):
    it.close()

print(f"Sample shape: {df_sample.shape}")
print(f"Total columns (from sample): {len(all_cols)}")
print("First 20 columns:", all_cols[:20])

Reading data from: ../data_raw\randhrs1992_2022v1.dta
File exists: True
Sample shape: (5000, 19880)
Total columns (from sample): 19880
First 20 columns: ['hhidpn', 's1hhidpn', 'r1mstat', 'r1mpart', 's1bmonth', 's1byear', 's1bdate', 's1bflag', 's1cohbyr', 's1hrsamp', 's1ahdsmp', 's1dmonth', 's1dyear', 's1ddate', 's1dsrc', 's1dtimtdth', 's1dage_y', 's1dage_m', 's1racem', 's1hispan']


In [24]:
df_sample.head()

,hhidpn,s1hhidpn,r1mstat,r1mpart,s1bmonth,s1byear,s1bdate,s1bflag,s1cohbyr,s1hrsamp,...,r8lbsatwlf,r9lbsatwlf,r10lbsatwlf,r11lbsatwlf,r12lbsatwlf,r13lbsatwlf,r14lbsatwlf,r15lbsatwlf,r16lbsatwlf,filever
0,1010,0.0,5.divorced,0.no,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X
1,2010,0.0,7.widowed,0.no,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,X
2,3010,3020.0,1.married,0.no,9.0,1938.0,-7778.0,0.mo/yr ok,3.hrs,"1.in samp,hrs92 resp b.1931-41",...,5.4,NaN,6.4,NaN,NaN,NaN,NaN,NaN,NaN,X
3,3020,3010.0,1.married,0.no,1.0,1936.0,-8752.0,0.mo/yr ok,3.hrs,"1.in samp,hrs92 resp b.1931-41",...,4.6,NaN,4.2,NaN,2.4,NaN,NaN,NaN,NaN,X
4,10001010,0.0,8.never married,0.no,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,2.8,NaN,4.8,NaN,NaN,NaN,NaN,NaN,X


In [25]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 19880 entries, hhidpn to filever
dtypes: category(13262), float32(104), float64(6473), int16(1), int32(5), int8(15), str(20)
memory usage: 314.9 MB


In [26]:
df_sample.describe(include="all").T.head(30)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hhidpn,5000.0,NaN,NaN,NaN,23069997.6868,8518173.313978,1010.0,16024017.5,21410515.0,32209275.75,37949040.0
s1hhidpn,4580.0,NaN,NaN,NaN,18946866.989083,11973308.084482,0.0,12212512.5,19286015.0,30821017.5,37949040.0
r1mstat,4580,7,1.married,3566,NaN,NaN,NaN,NaN,NaN,NaN,NaN
r1mpart,4580,2,0.no,4458,NaN,NaN,NaN,NaN,NaN,NaN,NaN
s1bmonth,3703.0,NaN,NaN,NaN,6.456657,3.40094,1.0,4.0,6.0,9.0,12.0
s1byear,3703.0,NaN,NaN,NaN,1936.712395,6.031448,1913.0,1933.0,1937.0,1940.0,1969.0
s1bdate,3703.0,NaN,NaN,NaN,-8326.063192,2204.402182,-17001.0,-9788.0,-8296.0,-7047.0,3361.0
s1bflag,3703,2,0.mo/yr ok,3699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
s1cohbyr,3703,8,3.hrs,2672,NaN,NaN,NaN,NaN,NaN,NaN,NaN
s1hrsamp,3703,2,"1.in samp,hrs92 resp b.1931-41",2548,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
missing_rate = df_sample.isna().mean().sort_values(ascending=False)
missing_rate.head(20)

r2moneyr      1.0
r2mealsr      1.0
s12ssiappm    1.0
r2shopr       1.0
s12ssirecy    1.0
s13ssirecy    1.0
s13ssiaply    1.0
s13ssiappy    1.0
r2walkre      1.0
r2eyecat      1.0
s8jlocca      1.0
s8jcocca      1.0
s9jlocca      1.0
s9jcocca      1.0
r8depwks      1.0
s11jlocca     1.0
s11jcocca     1.0
s12jlocca     1.0
s12jcocca     1.0
s2mealsr      1.0
dtype: float64

In [28]:
key_candidates = ["age", "ragender", "raeduc", "cesd", "hosp"]
for c in key_candidates:
    if c in df_sample.columns:
        print("\n===", c, "===")
        display(df_sample[c].value_counts(dropna=False).head(10))


=== ragender ===


ragender
2.female    2660
1.male      2340
Name: count, dtype: int64


=== raeduc ===


raeduc
3.high-school graduate    1616
1.lt high-school          1162
4.some college            1007
5.college and above        953
2.ged                      255
NaN                          7
Name: count, dtype: int64

In [ ]:
missing_report = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_rate")
)

missing_report.head(20)

,missing_rate
s12jcocca,1.0
s13jcocca,1.0
s4prsrc2,1.0
radendm10,1.0
radrecd11,1.0
r15ptypf4,1.0
r16jcocca,1.0
s16dcpct4,1.0
r15ptyp4,1.0
radendy7,1.0


In [ ]:
key_vars = [
    "age",
    "sex",
    "adl_any",
    "diabetes",
    "cesd_score"
]

for var in key_vars:
    if var in df.columns:
        print(f"\n==== {var} ====")
        print(df[var].value_counts(dropna=False).head())

In [ ]:
# ============================================
# 转换为长格式面板
# ============================================

def build_long_panel(df):
    """
    将宽格式的RAND HRS数据转换为长格式面板
    """
    waves = range(1, 17)
    out = []
    
    # 提取时间不变的变量
    gender = pd.to_numeric(df["ragender"], errors="coerce") if "ragender" in df.columns else pd.Series([np.nan] * len(df))
    edu = pd.to_numeric(df["raedyrs"], errors="coerce") if "raedyrs" in df.columns else pd.Series([np.nan] * len(df))
    
    for w in waves:
        age_col = f"r{w}agey_e"
        hosp_col = f"r{w}hosp"
        diab_col = f"r{w}diab"
        cesd_col = f"r{w}cesd" if w >= 2 else None
        adl_col = "r1adlw" if w == 1 else f"r{w}adl5a"
        
        # 创建当前wave的数据框
        tmp = pd.DataFrame({
            "hhidpn": df["hhidpn"],
            "wave": w,
            "gender": gender,
            "edu_years": edu,
            "age": pd.to_numeric(df[age_col], errors="coerce") if age_col in df.columns else np.nan,
            "hosp": pd.to_numeric(df[hosp_col], errors="coerce") if hosp_col in df.columns else np.nan,
            "diabetes": pd.to_numeric(df[diab_col], errors="coerce") if diab_col in df.columns else np.nan,
        })
        
        # 添加CESD（如果有）
        if cesd_col and cesd_col in df.columns:
            tmp["cesd"] = pd.to_numeric(df[cesd_col], errors="coerce")
        else:
            tmp["cesd"] = np.nan
            
        # 添加ADL
        if adl_col in df.columns:
            adl_val = pd.to_numeric(df[adl_col], errors="coerce")
            tmp["adl_any"] = (adl_val > 0).astype(float)
        else:
            tmp["adl_any"] = np.nan
        
        # ✅ 关键：只保留有实际访谈记录的波次（age有值）
        tmp = tmp[tmp["age"].notna()].copy()
        
        out.append(tmp)
    
    return pd.concat(out, ignore_index=True)

# 执行转换
print("Converting to long format...")
panel = build_long_panel(df)
print(f"Panel shape: {panel.shape}")
print(f"Unique individuals: {panel['hhidpn'].nunique()}")
print(f"Waves range: {int(panel['wave'].min())}-{int(panel['wave'].max())}")

Converting to long format...
Panel shape: (296197, 9)
Unique individuals: 45233
Waves range: 1-16


In [ ]:
if "age" in df.columns:
    plt.hist(df["age"].dropna(), bins=30)
    plt.title("Age Distribution")
    plt.xlabel("Age")
    plt.ylabel("Count")
    plt.show()

In [ ]:
binary_vars = ["sex", "adl_any", "diabetes"]

for var in binary_vars:
    if var in df.columns:
        df[var].value_counts(dropna=False).plot(kind="bar")
        plt.title(f"{var} Distribution")
        plt.show()

# Section 6 — Generate Longitudinal Clean Panel Dataset

This section constructs a person-wave panel dataset from the RAND HRS Longitudinal File.

Output:
- ../data_processed/notebook0_clean_panel.csv

This dataset will be used for:
- Baseline analytic extraction (Notebook1)
- Longitudinal impact evaluation (Notebook4)

In [ ]:
# ============================================
# 6. Generate Slim Longitudinal Panel (ULTRA MEMORY-SAFE)
#   - Read 1 row to get column names (old pandas safe)
#   - Auto-detect age-like stub (r{wave}*age*)
#   - Load only required columns
#   - Wide → Long
#   - Save: ../data_processed/notebook0_clean_panel.csv
# ============================================

import os
import re
import pandas as pd
import numpy as np

DATA_RAW = "../data_raw/randhrs1992_2022v1.dta"
DATA_DIR = "../data_processed"
OUT_PANEL = os.path.join(DATA_DIR, "notebook0_clean_panel.csv")
os.makedirs(DATA_DIR, exist_ok=True)

print("Step 1) Reading column names via iterator (read 1 row)...")
it = pd.read_stata(DATA_RAW, iterator=True)
tmp0 = it.read(1)  # some pandas versions cannot read(0); read(1) is safe
all_cols = [c.lower().strip() for c in tmp0.columns]
del tmp0
if hasattr(it, "close"):
    it.close()

print("Total columns detected:", len(all_cols))
print("First 20 columns:", all_cols[:20])

# ---------------------------------------------------------
# A) Auto-detect AGE-like wave variables: r{wave}...age...
#    Example matches: r1age, r2agey, r10agey, r3age_yrs ...
# ---------------------------------------------------------
age_like_cols = [c for c in all_cols if re.match(r"^r\d+.*age.*$", c)]
print("\n[Auto search] Age-like r{wave}*age* columns (first 40):", age_like_cols[:40])

def extract_stub_from_wavecol(colname: str) -> str:
    """
    Convert 'r10agey' -> 'agey'
    Convert 'r2age_yrs' -> 'age_yrs'
    """
    m = re.match(r"^r\d+(.*)$", colname)
    return m.group(1) if m else ""

age_like_stubs = sorted({extract_stub_from_wavecol(c) for c in age_like_cols if extract_stub_from_wavecol(c)})
print("[Auto search] Candidate AGE stubs:", age_like_stubs[:40])

# Prefer common patterns if present
preferred_age_stubs = ["agey", "age", "age_yrs", "ageyrs", "ageyr", "age_years", "ageyear"]
auto_age_stub = None
for s in preferred_age_stubs:
    if s in age_like_stubs:
        auto_age_stub = s
        break
if auto_age_stub is None and len(age_like_stubs) > 0:
    auto_age_stub = age_like_stubs[0]  # fallback: first candidate

print("[Auto search] Selected AGE stub:", auto_age_stub)

# ---------------------------------------------------------
# B) Choose stubs needed for Notebook4
#    NOTE: We'll insert the auto-detected age stub (if any)
# ---------------------------------------------------------
stub_candidates = [
    # age-related (auto)
    # (auto_age_stub will be inserted if found)
    "sex", "gender",
    "edu_years", "edu",
    "cesd", "adl_any", "adl",
    "hosp", "hospital",
    "diabetes", "hypertension", "heart", "stroke"
]

if auto_age_stub is not None:
    # put age stub at the front
    stub_candidates = [auto_age_stub] + stub_candidates

print("\nStub candidates (final):", stub_candidates)

def wide_cols_for_stub(cols, stub):
    # matches r1<stub>, r2<stub> ... exactly
    pat = re.compile(rf"^r(\d+){re.escape(stub)}$")
    return [c for c in cols if pat.match(c)]

available_stubs = {}
for stub in stub_candidates:
    cols_stub = wide_cols_for_stub(all_cols, stub)
    if cols_stub:
        available_stubs[stub] = cols_stub

print("Found stubs:", sorted(available_stubs.keys()))
if not available_stubs:
    raise RuntimeError(
        "No r{wave}{stub} variables found for chosen stubs. "
        "We must adjust stub names to match your RAND file."
    )

# ---------------------------------------------------------
# C) Optional static (non-wave) vars
# ---------------------------------------------------------
static_candidates = ["ragender", "raracem", "rarace", "rahispan"]
static_keep = [c for c in static_candidates if c in all_cols]

# ---------------------------------------------------------
# D) Final columns to read (ONLY these)
# ---------------------------------------------------------
use_cols = ["hhidpn"] + sorted({c for cols in available_stubs.values() for c in cols})
use_cols += static_keep
use_cols = sorted(set(use_cols))

print(f"\nStep 2) Loading ONLY selected columns: {len(use_cols)} columns...")
df = pd.read_stata(DATA_RAW, columns=use_cols, convert_categoricals=False)
df.columns = df.columns.str.lower().str.strip()

if "hhidpn" not in df.columns:
    raise ValueError("hhidpn not found after selective load.")
df["hhidpn"] = df["hhidpn"].astype(str).str.strip()

print("Loaded slim df shape:", df.shape)

# ---------------------------------------------------------
# E) Wide → Long (melt per stub then merge)
# ---------------------------------------------------------
panel = None
for stub, wide_cols in available_stubs.items():
    wide_cols = [c for c in wide_cols if c in df.columns]  # safety
    tmp = df[["hhidpn"] + wide_cols].copy()

    long = tmp.melt(id_vars=["hhidpn"], var_name="wide_col", value_name=stub)
    long["wave"] = long["wide_col"].str.extract(r"^r(\d+)", expand=False).astype(int)
    long = long.drop(columns=["wide_col"])

    # clean RAND missing codes like ".D"
    if long[stub].dtype == "object":
        long[stub] = long[stub].replace({r"^\.\w+$": np.nan}, regex=True)

    # numeric
    long[stub] = pd.to_numeric(long[stub], errors="coerce")

    if panel is None:
        panel = long
    else:
        panel = panel.merge(long, on=["hhidpn", "wave"], how="outer")

    print(f"Added {stub:<12} | panel now: {panel.shape}")

# ---------------------------------------------------------
# F) Merge static vars (if any)
# ---------------------------------------------------------
if static_keep:
    static = df[["hhidpn"] + static_keep].copy()
    for c in static_keep:
        if static[c].dtype == "object":
            static[c] = static[c].replace({r"^\.\w+$": np.nan}, regex=True)
        static[c] = pd.to_numeric(static[c], errors="coerce")
    panel = panel.merge(static, on="hhidpn", how="left")
    print("Merged static vars:", static_keep)

panel = panel.sort_values(["hhidpn", "wave"]).reset_index(drop=True)

print("\nDONE")
print("Panel shape:", panel.shape)
print("Unique individuals:", panel["hhidpn"].nunique())
print("Wave range:", int(panel["wave"].min()), "to", int(panel["wave"].max()))
print("Columns:", panel.columns.tolist())

panel.to_csv(OUT_PANEL, index=False)
print("Saved panel dataset to:", OUT_PANEL)

Step 1) Reading column names via iterator (read 1 row)...
Total columns detected: 19880
First 20 columns: ['hhidpn', 's1hhidpn', 'r1mstat', 'r1mpart', 's1bmonth', 's1byear', 's1bdate', 's1bflag', 's1cohbyr', 's1hrsamp', 's1ahdsmp', 's1dmonth', 's1dyear', 's1ddate', 's1dsrc', 's1dtimtdth', 's1dage_y', 's1dage_m', 's1racem', 's1hispan']

[Auto search] Age-like r{wave}*age* columns (first 40): ['r1agem_b', 'r2agem_b', 'r3agem_b', 'r4agem_b', 'r5agem_b', 'r6agem_b', 'r7agem_b', 'r8agem_b', 'r9agem_b', 'r10agem_b', 'r11agem_b', 'r12agem_b', 'r13agem_b', 'r14agem_b', 'r15agem_b', 'r16agem_b', 'r1agem_e', 'r2agem_e', 'r3agem_e', 'r4agem_e', 'r5agem_e', 'r6agem_e', 'r7agem_e', 'r8agem_e', 'r9agem_e', 'r10agem_e', 'r11agem_e', 'r12agem_e', 'r13agem_e', 'r14agem_e', 'r15agem_e', 'r16agem_e', 'r1agem_m', 'r2agem_m', 'r3agem_m', 'r4agem_m', 'r5agem_m', 'r6agem_m', 'r7agem_m', 'r8agem_m']
[Auto search] Candidate AGE stubs: ['agem_b', 'agem_e', 'agem_m', 'agey_b', 'agey_e', 'agey_m', 'dadage', 'dbst

## EDA Summary

Key observations:

- Reviewed variable structure and column types
- Examined missingness patterns
- Inspected distribution of age, sex, ADL limitation, diabetes, and depression score
- Identified potential missing data issues to address in Notebook 1

Next Step:

Proceed to Notebook 1 to:

- Define baseline cohort (Age ≥ 65)
- Clean variables
- Construct analytic dataset
- Save `notebook1_baseline_analytic.csv`